# Домашка. MatrixVisualizer (7 баллов)

Кто не понимает контекст, смотрим первый семинар по интерпретации :)
Для выполнения задания РЕКОМЕНДУЕТСЯ пользоваться LLM (с умом!),
    а также посмотреть семинар по bokeh и (дополнительно) HDBSCAN.

Вам нужно написать класс, который визуализирует произвольную матрицу / датафрейм чисел в формате интерактивной heatmap с помощью библиотеки bokeh. Для удобства представим, что на вход приходит матрица размера N_features x K, то есть, своего рода эмбеддинги признаков (например, полученные путем агрегирования инфы из `model.trees_to_dataframe()`). Сами решите, в каком формате вы подаете сами данные (np.ndarray + отдельно лейблы осей Х, Y или же датафрейм или же что угодно еще). Класс нужно реализовать универсальным образом, то есть не затачиваясь конкретно на то, что вам придут именно признаки, однако в формулировке задания в целях удобочитаемости я буду писать "признаки", где имею в виду строки или же подписи к оси Y. 

Итак, вам нужно визуализировать матрицу согласно следующим требованиям:
- heatmap-style визуализация (см. семинар)
- Если N_features (в общем случае кол-во строк матрицы) велико, то в первоначальной картинке "признаков" не видно.
      Они должны становиться видны при достаточно крупном зуме (решайте сами каком), чтобы картинка была читаема.
      Соответственно, должен быть реализован механизм зума (через любой инструмент)
- График должен быть раскрашен с помощью QuantileTransformer'a:
```python
a = np.array([1, 2, 0, 4, 10, -1])
b = QT(a, num_quantiles=6)
# b = np.array([0.4, 0.2, 0.6, 0.8, 1., 0.])
```
    (то есть цвета для раскраски формируются по b, а не по a)

- Палитра 'coolwarm'. Должен быть colorbar + числовые подписи к его цветам (хотя бы несколько основных)
- Где-нибудь рядышком должен быть виджет выпадающего окна, в котором можно выбрать, "по какой оси" делать покраску:
    - по строкам (то есть у каждого "признака" свои квантили; вкл. по умолчанию)
    - по столбцам
    - глобальная (то есть, квантиль 1. получает максимальное число в матрице, а 0. -- минимальное)<br>
- После выбора "оси", раскраска должна перерисовываться на лету
- Ваши "признаки" должны быть отсортированы согласно кластеризации HDBSCAN (кормим на вход эмбеддинги "признаков",
      желательно сразу в построчной квантильной форме, которую считали для покраски по умолчанию).
      Не нужно строить иерархии, можно просто отсортировать по номеру кластера (не забудьте отсортировать и "подписи").
      Нам важно только чтобы похожие "признаки" были рядом. Расположение кластеров относительно друг друга не имеет значения
- Должна быть возможность "выбрать" какие-то признаки (например, инструментом Tap / Выделением),
      после чего они должны появляться в отдельном текстовом окошечке с накинутыми на них кавычками
      и кнопкой Копировать рядом для удобства.
      Это может быть полезно, например, для того, чтобы быстро выкинуть из выборки "признаки", которые вам кажутся
      неудачными при анализе данной матрицы.
- Весь интерактив с виджетами должен быть реализован строго БЕЗ js-коллбэков (можно только для копирования текста в буфер),
    только python-коллбэки с поднятием локального
    bokeh server в ноутбуке (конструкция def app(doc): ... + show(app), см. семинар по bokeh, HDBSCAN)
- Придумайте удобный вам формат принятия данных на вход для того чтобы подписать оси X, Y, (labels, tick_labels)
- Попросите LLM сгенерировать какую-нибудь матрицу, которая наглядно бы демонстрировала, что вы справились с заданием,
      и запустите на ней вашу визуализацию

**! Если ваша визуализация не заработает (буду запускать локально с вашими версиями библиотек; обновитесь по возможности до последних версий hdbscan, bokeh), будет 0 баллов, так что перед отправкой убедитесь, что оно работает как надо**

Ниже дан ориентир: (там выполнены не все пункты, выполняйте свои)

## Удачи!

In [1]:
import inspect
import warnings

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.colors as mcolors

from sklearn.preprocessing import QuantileTransformer
from bokeh.io import output_notebook, show
from bokeh.layouts import row, column
from bokeh.models import (
    BasicTicker,
    Button,
    ColorBar,
    ColumnDataSource,
    CustomJS,
    Div,
    FixedTicker,
    HoverTool,
    LinearColorMapper,
    PrintfTickFormatter,
    Range1d,
    Select,
    TextAreaInput,
)
from bokeh.plotting import figure

warnings.filterwarnings("ignore", category=FutureWarning)
output_notebook(hide_banner=True)

import hdbscan


def coolwarm_palette(n_colors: int = 256) -> list[str]:
    cmap = matplotlib.colormaps.get_cmap("coolwarm").resampled(n_colors)
    return [mcolors.to_hex(cmap(i)) for i in range(cmap.N)]


class MatrixVisualizer:
    def __init__(
        self,
        data: pd.DataFrame | np.ndarray,
        row_labels: list[str] | None = None,
        col_labels: list[str] | None = None,
        title: str = "MatrixVisualizer",
        label_zoom_threshold: int = 28,
        hdbscan_min_cluster_size: int | None = None,
    ) -> None:
        if isinstance(data, pd.DataFrame):
            df = data.copy()
        else:
            arr = np.asarray(data, dtype=float)
            if arr.ndim != 2:
                raise ValueError("data must be a 2D matrix")
            if row_labels is None:
                row_labels = [f"feature_{i:03d}" for i in range(arr.shape[0])]
            if col_labels is None:
                col_labels = [f"col_{j:02d}" for j in range(arr.shape[1])]
            df = pd.DataFrame(arr, index=list(row_labels), columns=list(col_labels))

        if row_labels is not None:
            df.index = list(row_labels)
        if col_labels is not None:
            df.columns = list(col_labels)

        self.df_raw = df.astype(float)
        self.row_labels = self.df_raw.index.astype(str).tolist()
        self.col_labels = self.df_raw.columns.astype(str).tolist()
        self.title = title
        self.label_zoom_threshold = int(label_zoom_threshold)
        self.palette = coolwarm_palette(256)
        self.cluster_backend = "hdbscan"

        self.transforms = {
            "По строкам": self._quantile_by_axis(self.df_raw.values, axis=1),
            "По столбцам": self._quantile_by_axis(self.df_raw.values, axis=0),
            "Глобально": self._quantile_global(self.df_raw.values),
        }

        order, cluster_labels = self._cluster_order(
            self.transforms["По строкам"],
            min_cluster_size=hdbscan_min_cluster_size,
        )

        self.order = order
        self.cluster_labels = cluster_labels[order]
        self.df_raw = self.df_raw.iloc[order]
        self.row_labels = [self.row_labels[i] for i in order]
        self.transforms = {name: matrix[order] for name, matrix in self.transforms.items()}
        self.n_rows, self.n_cols = self.df_raw.shape

        for name, matrix in self.transforms.items():
            if not (np.nanmin(matrix) >= -1e-12 and np.nanmax(matrix) <= 1 + 1e-12):
                raise ValueError(f"Quantile matrix '{name}' is outside [0, 1]")

    @staticmethod
    def _quantile_vector(values: np.ndarray) -> np.ndarray:
        values = np.asarray(values, dtype=float).reshape(-1, 1)
        n_quantiles = min(max(len(values), 2), 1000)
        transformer = QuantileTransformer(
            n_quantiles=n_quantiles,
            output_distribution="uniform",
            random_state=0,
        )
        return transformer.fit_transform(values).ravel()

    def _quantile_by_axis(self, matrix: np.ndarray, axis: int) -> np.ndarray:
        out = np.zeros_like(matrix, dtype=float)
        if axis == 1:
            for row_idx in range(matrix.shape[0]):
                out[row_idx, :] = self._quantile_vector(matrix[row_idx, :])
        elif axis == 0:
            for col_idx in range(matrix.shape[1]):
                out[:, col_idx] = self._quantile_vector(matrix[:, col_idx])
        else:
            raise ValueError("axis must be 0 (columns) or 1 (rows)")
        return out

    @staticmethod
    def _quantile_global(matrix: np.ndarray) -> np.ndarray:
        flat = matrix.reshape(-1, 1)
        n_quantiles = min(max(flat.shape[0], 2), 1000)
        transformer = QuantileTransformer(
            n_quantiles=n_quantiles,
            output_distribution="uniform",
            random_state=0,
        )
        return transformer.fit_transform(flat).reshape(matrix.shape)

    @staticmethod
    def _make_hdbscan_estimator(min_cluster_size: int):
        signature = inspect.signature(hdbscan.HDBSCAN.__init__)
        supported = set(signature.parameters)
        kwargs = {
            "min_cluster_size": min_cluster_size,
            "min_samples": 2,
            "allow_single_cluster": True,
            "copy": True,
        }
        filtered_kwargs = {key: value for key, value in kwargs.items() if key in supported}
        return hdbscan.HDBSCAN(**filtered_kwargs)

    def _cluster_order(self, features: np.ndarray, min_cluster_size: int | None = None):
        n_rows = features.shape[0]
        min_cluster_size = min_cluster_size or max(4, min(20, n_rows // 10))
        clusterer = self._make_hdbscan_estimator(min_cluster_size)
        labels = clusterer.fit_predict(features)

        row_means = features.mean(axis=1)
        if np.all(labels == -1):
            order = np.argsort(row_means)
        else:
            order = np.lexsort((row_means, labels))
        return order, labels

    def _make_long_source_dict(self, mode: str) -> dict[str, np.ndarray]:
        quant_matrix = self.transforms[mode]
        raw_matrix = self.df_raw.values
        xs, ys = np.meshgrid(np.arange(self.n_cols), np.arange(self.n_rows))

        return {
            "x": xs.ravel().astype(float),
            "y": ys.ravel().astype(float),
            "x_idx": xs.ravel().astype(int),
            "y_idx": ys.ravel().astype(int),
            "col_label": np.repeat(np.array(self.col_labels, dtype=object)[None, :], self.n_rows, axis=0).ravel(),
            "row_label": np.repeat(np.array(self.row_labels, dtype=object)[:, None], self.n_cols, axis=1).ravel(),
            "raw_value": raw_matrix.ravel().astype(float),
            "q_value": quant_matrix.ravel().astype(float),
            "cluster": np.repeat(self.cluster_labels[:, None], self.n_cols, axis=1).ravel().astype(int),
        }

    def create_layout(self):
        initial_mode = "По строкам"
        source = ColumnDataSource(self._make_long_source_dict(initial_mode))

        x_range = Range1d(-0.5, self.n_cols - 0.5, bounds=(-0.5, self.n_cols - 0.5))
        y_range = Range1d(self.n_rows - 0.5, -0.5, bounds=(-0.5, self.n_rows - 0.5))
        color_mapper = LinearColorMapper(palette=self.palette, low=0.0, high=1.0)

        plot = figure(
            title=f"{self.title} — сортировка строк через {self.cluster_backend}",
            width=1120,
            height=720,
            x_range=x_range,
            y_range=y_range,
            tools="pan,wheel_zoom,box_zoom,reset,save,tap,box_select,lasso_select",
            active_scroll="wheel_zoom",
            toolbar_location="above",
        )
        plot.min_border_left = 220
        plot.grid.grid_line_color = None
        plot.axis.axis_line_color = "#666666"
        plot.axis.major_tick_line_color = "#666666"
        plot.axis.minor_tick_line_color = None
        plot.yaxis.major_label_text_font_size = "9pt"
        plot.xaxis.major_label_text_font_size = "9pt"
        plot.xaxis.major_label_orientation = 0.9

        plot.rect(
            x="x",
            y="y",
            width=1,
            height=1,
            source=source,
            fill_color={"field": "q_value", "transform": color_mapper},
            line_color=None,
            nonselection_fill_alpha=0.96,
            selection_line_color="black",
            selection_line_width=1.5,
        )

        hover = HoverTool(
            tooltips=[
                ("Строка", "@row_label"),
                ("Столбец", "@col_label"),
                ("Исходное значение", "@raw_value{0.000}"),
                ("Квантиль", "@q_value{0.000}"),
                ("Кластер HDBSCAN", "@cluster"),
            ]
        )
        plot.add_tools(hover)
        plot.toolbar.active_inspect = [hover]

        plot.xaxis.ticker = FixedTicker(ticks=list(range(self.n_cols)))
        plot.xaxis.major_label_overrides = {idx: label for idx, label in enumerate(self.col_labels)}
        plot.yaxis.ticker = FixedTicker(ticks=[])
        plot.yaxis.major_label_overrides = {idx: label for idx, label in enumerate(self.row_labels)}

        color_bar = ColorBar(
            color_mapper=color_mapper,
            ticker=BasicTicker(desired_num_ticks=6),
            formatter=PrintfTickFormatter(format="%.2f"),
            label_standoff=8,
        )
        plot.add_layout(color_bar, "right")

        mode_select = Select(
            title="Раскраска квантилями",
            value=initial_mode,
            options=["По строкам", "По столбцам", "Глобально"],
            width=320,
        )
        selection_info = Div(text="Выделено строк: 0", width=320)
        help_text = Div(
            text=(
                "Выбирайте ячейки через Tap / Box Select / Lasso Select. "
                "Справа появятся уникальные названия строк в кавычках. "
                "Подписи строк скрыты до тех пор, пока вы не приблизите матрицу достаточно сильно."
            ),
            width=320,
        )
        selected_box = TextAreaInput(title="Выбранные строки", value="", rows=18, cols=36, width=340)
        copy_button = Button(label="Копировать", button_type="primary", width=150)
        clear_button = Button(label="Очистить выбор", button_type="default", width=150)

        copy_button.js_on_click(
            CustomJS(
                args={"text_area": selected_box},
                code="""
                    const text = text_area.value || '';
                    if (navigator.clipboard && window.isSecureContext) {
                        navigator.clipboard.writeText(text);
                    } else {
                        const area = document.createElement('textarea');
                        area.value = text;
                        document.body.appendChild(area);
                        area.focus();
                        area.select();
                        document.execCommand('copy');
                        document.body.removeChild(area);
                    }
                """,
            )
        )

        def refresh_y_ticks() -> None:
            low = min(plot.y_range.start, plot.y_range.end)
            high = max(plot.y_range.start, plot.y_range.end)
            visible_rows = [idx for idx in range(self.n_rows) if low - 0.5 <= idx <= high + 0.5]
            if len(visible_rows) <= self.label_zoom_threshold:
                plot.yaxis.ticker = FixedTicker(ticks=visible_rows)
            else:
                plot.yaxis.ticker = FixedTicker(ticks=[])

        def update_selection(attr, old, new) -> None:
            indices = source.selected.indices
            if not indices:
                selected_box.value = ""
                selection_info.text = "Выделено строк: 0"
                return

            selected_y = np.array(source.data["y_idx"])[indices]
            ordered_unique_rows: list[str] = []
            seen: set[int] = set()
            for row_idx in selected_y[np.argsort(selected_y, kind="stable")]:
                row_idx = int(row_idx)
                if row_idx not in seen:
                    seen.add(row_idx)
                    ordered_unique_rows.append(self.row_labels[row_idx])

            selected_box.value = ",\n".join(f'"{name}"' for name in ordered_unique_rows)
            selection_info.text = f"Выделено строк: {len(ordered_unique_rows)}"

        def update_y_zoom(attr, old, new) -> None:
            refresh_y_ticks()

        def recolor(attr, old, new) -> None:
            source.data = self._make_long_source_dict(new)
            source.selected.indices = []
            selected_box.value = ""
            selection_info.text = "Выделено строк: 0"

        def clear_selection() -> None:
            source.selected.indices = []
            selected_box.value = ""
            selection_info.text = "Выделено строк: 0"

        source.selected.on_change("indices", update_selection)
        plot.y_range.on_change("start", update_y_zoom)
        plot.y_range.on_change("end", update_y_zoom)
        mode_select.on_change("value", recolor)
        clear_button.on_click(clear_selection)
        refresh_y_ticks()

        controls = column(
            mode_select,
            selection_info,
            help_text,
            selected_box,
            row(copy_button, clear_button),
            width=360,
        )
        return row(plot, controls)

    def show(
        self,
        notebook_url: str = "localhost:8888",
        vscode_webview_origin: str | None = None,
        port: int = 0,
    ) -> None:
        def normalize_origin(origin: str) -> str:
            origin = str(origin).strip().rstrip("/")
            if "://" in origin:
                origin = origin.split("://", 1)[1]
            return origin

        def app(doc):
            layout = self.create_layout()
            doc.add_root(layout)
            doc.title = self.title

        if vscode_webview_origin:
            clean_origin = normalize_origin(vscode_webview_origin)

            def vscode_notebook_url(server_port):
                if server_port is None:
                    return clean_origin
                return f"http://localhost:{server_port}"

            show(app, notebook_url=vscode_notebook_url, port=port)
        else:
            show(app, notebook_url=notebook_url, port=port)


def make_llm_generated_demo_matrix(random_state: int = 42) -> pd.DataFrame:
    """
    Synthetic matrix explicitly generated for the homework demo.
    The goal is not realism of data collection, but a clear visual check that:
    1) HDBSCAN groups similar rows,
    2) different recoloring modes behave differently,
    3) zoom-dependent row labels are useful on a tall matrix.
    """
    rng = np.random.default_rng(random_state)

    n_clusters = 6
    rows_per_cluster = 32
    n_cols = 12
    x = np.linspace(0, 2 * np.pi, n_cols)

    base_patterns = np.vstack([
        1.25 * np.sin(x),
        1.00 * np.cos(1.5 * x + 0.3),
        np.linspace(-1.4, 1.3, n_cols),
        np.r_[np.full(n_cols // 2, 1.1), np.full(n_cols - n_cols // 2, -1.2)],
        np.exp(-0.5 * ((np.arange(n_cols) - 3.5) / 1.4) ** 2) * 2.0 - 0.5,
        np.exp(-0.5 * ((np.arange(n_cols) - 8.0) / 1.6) ** 2) * 2.1 - 0.6,
    ])

    rows = []
    labels = []
    readable_prefixes = [
        "billing",
        "fraud",
        "engagement",
        "quality",
        "latency",
        "retention",
    ]

    for cluster_id in range(n_clusters):
        base = base_patterns[cluster_id]
        for local_idx in range(rows_per_cluster):
            noise = rng.normal(0, 0.18, size=n_cols)
            scale = rng.uniform(0.85, 1.20)
            shift = rng.normal(0, 0.12)
            local_spike = np.zeros(n_cols)
            local_spike[rng.integers(0, n_cols)] = rng.normal(0, 0.8)
            row = scale * base + shift + noise + 0.25 * local_spike
            rows.append(row)
            labels.append(f"{readable_prefixes[cluster_id]}_feature_{local_idx:02d}")

    for outlier_idx in range(12):
        rows.append(rng.normal(0, 1.1, size=n_cols))
        labels.append(f"outlier_feature_{outlier_idx:02d}")

    matrix = np.asarray(rows)
    columns = [f"emb_{idx:02d}" for idx in range(n_cols)]
    return pd.DataFrame(matrix, index=labels, columns=columns)


demo_df = make_llm_generated_demo_matrix(random_state=42)
visualizer = MatrixVisualizer(
    demo_df,
    title="Homework 9 — MatrixVisualizer",
    label_zoom_threshold=26,
)

print(f"Matrix shape: {demo_df.shape}")
print("Controls:")
print("  • wheel / box zoom to reveal row labels")
print("  • dropdown to switch row / column / global quantile coloring")
print("  • tap / box / lasso select to collect row names on the right")
print("  • copy button uses the only JS callback in the notebook, as allowed")

visualizer.show(
    vscode_webview_origin="00p928bfduleahi3pide9osn0c06ve77dggsprh2mpv0i2qlhjn6"
)


Matrix shape: (204, 12)
Controls:
  • wheel / box zoom to reveal row labels
  • dropdown to switch row / column / global quantile coloring
  • tap / box / lasso select to collect row names on the right
  • copy button uses the only JS callback in the notebook, as allowed
